# Stage 1 - Phase 0: prepare DLC-2021Builds the shared Stage 1 artefacts every later notebook depends on:1. scan the DLC-2021 `or` / `re` subsets into the **common Stage 1 manifest**   (`cc` / `cg` are ignored),2. report broken videos and the class / resolution / FPS / duration   distributions,3. create the **reproducible video-level split** and write it to CSV.Two rules this notebook exists to enforce:* the split is decided at **video level, before any frame or patch is  extracted**, so frames of one video can never appear in both train and  validation;* resolution, FPS, duration and codec are **diagnostics only**. They are strong  shortcuts for this task and are never fed to a classifier.`source_video_id`, `capture_device` and `display_device` stay empty forDLC-2021 because the dataset exposes no verifiable pairing here. They are thehook for the planned paired CCD re-recordings; nothing is guessed.

## 1. Setup

In [ ]:
from __future__ import annotationsimport sysfrom pathlib import Pathimport numpy as npimport pandas as pdimport torchimport yaml# The package is expected to be installed with `pip install -e .` from the# repository root. The fallback keeps a fresh clone usable without installing.try:    import blackbox_detection  # noqa: F401except ModuleNotFoundError:    _root = Path.cwd()    while _root != _root.parent and not (_root / "pyproject.toml").is_file():        _root = _root.parent    sys.path.insert(0, str(_root / "src"))from blackbox_detection.utils import seed_everything, setup_loggerprint("torch", torch.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
from blackbox_detection.stage1 import (    SplitConfig,    ValidationSubsetSpec,    apply_split,    broken_videos,    build_validation_subsets,    drop_broken_videos,    load_manifest,    load_split,    make_video_level_split,    manifest_summary,    save_manifest,    save_split,    scan_dlc2021,    split_summary,)logger = setup_logger("stage1.prepare_dlc2021")

## 2. Paths

In [ ]:
REPO_ROOT = Path.cwd()while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").is_file():    REPO_ROOT = REPO_ROOT.parentCONFIG_DIR = REPO_ROOT / "configs" / "stage1"OUTPUT_ROOT = REPO_ROOT / "outputs" / "stage1"OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)print("repo   :", REPO_ROOT)print("configs:", CONFIG_DIR)print("outputs:", OUTPUT_ROOT)

## 3. ConfigEdit `configs/stage1/dlc2021.yaml`, or set `DLC_ROOT` below for a local path.

In [ ]:
DATA_CONFIG = yaml.safe_load((CONFIG_DIR / "dlc2021.yaml").read_text(encoding="utf-8"))DLC_ROOT = DATA_CONFIG["dataset"]["root"]# Machine-specific path: set it here when it is not in the config.# DLC_ROOT = r"D:/datasets/DLC-2021"if DLC_ROOT is None:    raise ValueError(        "Set dataset.root in configs/stage1/dlc2021.yaml or assign DLC_ROOT here."    )DLC_ROOT = Path(DLC_ROOT)MANIFEST_PATH = REPO_ROOT / DATA_CONFIG["paths"]["manifest"]SPLIT_PATH = REPO_ROOT / DATA_CONFIG["paths"]["split"]SEED = int(DATA_CONFIG["split"]["seed"])seed_everything(SEED, deterministic=True, strict=False)print("DLC-2021 root:", DLC_ROOT)print("manifest ->", MANIFEST_PATH)print("split    ->", SPLIT_PATH)

## 4. Data### 4.1 Scan into the common Stage 1 manifest

In [ ]:
dataset_config = DATA_CONFIG["dataset"]manifest = scan_dlc2021(    DLC_ROOT,    dataset=dataset_config["name"],    scene_type=dataset_config["scene_type"],    label_directories=dataset_config["label_directories"],    probe_metadata=dataset_config["probe_metadata"],    verify_decode=dataset_config["verify_decode"],)print("rows:", len(manifest))manifest.head()

### 4.2 Broken videos

In [ ]:
broken = broken_videos(manifest)print(f"{len(broken)} broken video(s)")if len(broken):    display(broken)if dataset_config["drop_broken"]:    manifest = drop_broken_videos(manifest)    print("usable videos:", len(manifest))

### 4.3 Class counts and diagnostic distributionsAnalysis only - none of these become model inputs.

In [ ]:
print(manifest["label"].value_counts().to_string())print()display(manifest_summary(manifest))

In [ ]:
resolution = (    manifest.assign(        resolution=manifest["width"].astype(str) + "x" + manifest["height"].astype(str)    )    .groupby(["resolution", "label"])    .size()    .unstack(fill_value=0))print("Resolution distribution per class:")display(resolution)print("FPS distribution per class:")display(manifest.groupby("label")["fps"].describe()[["count", "min", "50%", "max"]])print("Duration (s) distribution per class:")display(manifest.groupby("label")["duration_sec"].describe()[["count", "min", "50%", "max"]])print("Codec distribution per class:")display(manifest.groupby(["codec", "label"]).size().unstack(fill_value=0))

Read the tables above as a **shortcut risk report**. If ORIGINAL andRERECORDED occupy disjoint resolution, FPS or codec ranges, a model can reach ahigh DLC-2021 score without learning anything about recapture. That is exactlywhat the controlled VAL-B subset in section 7 is for.

### 4.4 Source grouping`source_video_id` is empty for DLC-2021, on purpose: an invented grouping wouldsilently corrupt the group-aware split once real paired CCD data arrives.

In [ ]:
known_groups = manifest.loc[manifest["source_video_id"] != "", "source_video_id"]print("videos with a known source_video_id:", len(known_groups))print("distinct source groups:", known_groups.nunique())

## 5. ModelNot applicable in Phase 0.

## 6. TrainingNot applicable in Phase 0.

## 7. Validation### 7.1 Video-level split`strategy: auto` resolves to a stratified split here and switches togroup-aware automatically once `source_video_id` is populated.

In [ ]:
split_config = SplitConfig(    val_size=float(DATA_CONFIG["split"]["val_size"]),    seed=SEED,    strategy=DATA_CONFIG["split"]["strategy"],    stratify_columns=tuple(DATA_CONFIG["split"]["stratify_columns"]),    group_column=DATA_CONFIG["split"]["group_column"],)split = make_video_level_split(manifest, config=split_config)display(split_summary(split))# Reproducibility check: the same seed must yield the same split.assert split.equals(make_video_level_split(manifest, config=split_config))print("split is reproducible for seed", SEED)

In [ ]:
splits = apply_split(manifest, split)train_manifest, val_manifest = splits["train"], splits["val"]print("train videos:", len(train_manifest), "| val videos:", len(val_manifest))assert not set(train_manifest["video_id"]) & set(val_manifest["video_id"])print("no video appears in both splits")

### 7.2 VAL-B: controlled subset diagnosticVAL-A is the full stratified validation set above. VAL-B restricts validation toa controlled slice (high resolution only) to expose a resolution shortcut. It isreported as *unusable*, rather than silently trusted, when it holds too fewvideos per class.

In [ ]:
subset_specs = [    ValidationSubsetSpec(        name=spec["name"],        column=spec["column"],        minimum=spec.get("minimum"),        maximum=spec.get("maximum"),        min_per_class=int(spec.get("min_per_class", 20)),        description=spec.get("description", ""),    )    for spec in DATA_CONFIG.get("validation_subsets", [])]for name, result in build_validation_subsets(val_manifest, subset_specs).items():    status = "usable" if result.usable else f"NOT usable ({result.reason})"    print(f"{name}: {len(result.video_ids)} video(s) | {result.class_counts} | {status}")

## 8. ResultsSanity checks that later notebooks assume.

In [ ]:
checks = {    "manifest rows": len(manifest),    "labels": sorted(manifest["label"].unique()),    "datasets": sorted(manifest["dataset"].unique()),    "scene types": sorted(manifest["scene_type"].unique()),    "synthetic rows": int(manifest["is_synthetic"].sum()),    "unique video_id": int(manifest["video_id"].nunique()) == len(manifest),    "all readable": bool(manifest["is_readable"].all()),    "train/val": (len(train_manifest), len(val_manifest)),}for key, value in checks.items():    print(f"{key:>18}: {value}")assert checks["unique video_id"], "video_id must be unique"# DLC-2021 recaptures are physical, not digitally simulated.assert int(manifest["is_synthetic"].sum()) == 0

## 9. SaveThe manifest and split CSVs are the fixed inputs of every Stage 1 experiment.

In [ ]:
save_manifest(manifest, MANIFEST_PATH)save_split(split, SPLIT_PATH)# Round-trip check.assert load_manifest(MANIFEST_PATH)["video_id"].tolist() == manifest["video_id"].tolist()assert load_split(SPLIT_PATH)["split"].tolist() == split["split"].tolist()print("manifest ->", MANIFEST_PATH)print("split    ->", SPLIT_PATH)print()print("Next: scripts/stage1/01_train_videomaev2_b.ipynb")